# S5 Causal Masked Reconstruction Colab

This notebook trains a causal `S5` encoder with masked signal reconstruction on Utah-array cache shards stored in Google Drive.

Default design:

- reconstruct normalized raw patch values in `TX` space
- causal `S5` backbone with session-keyed token read-in / read-out boundaries
- patch-level masking by default with `patch_size=4`, `patch_stride=2`
- contiguous masked spans with a fixed overall mask ratio
- same held-out `Brain2Text25` frozen phoneme probe workflow used in the other `s5` notebooks


In [1]:
# Mount Drive and resolve cache / output roots.

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
cache_candidates = [
    DRIVE_ROOT / "utah_ssl" / "data" / "cache_v1",
]

CACHE_ROOT = next((p for p in cache_candidates if p.exists()), cache_candidates[0])
OUTPUT_ROOT = DRIVE_ROOT / "utah_ssl" / "outputs" / "ssl_experiments" / "masked_reconstruction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT :", DRIVE_ROOT)
print("CACHE_ROOT :", CACHE_ROOT, "| exists:", CACHE_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT, "| exists:", OUTPUT_ROOT.exists())

if CACHE_ROOT.exists():
    datasets = sorted(p.name for p in CACHE_ROOT.iterdir() if p.is_dir())
    print("datasets:", datasets)
else:
    print("cache candidates checked:")
    for path in cache_candidates:
        print(" -", path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT : /content/drive/MyDrive
CACHE_ROOT : /content/drive/MyDrive/utah_ssl/data/cache_v1 | exists: True
OUTPUT_ROOT: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction | exists: True
datasets: ['000950', 'brain2text24', 'brain2text25', 'motor_data', 'plug_n_play', 'unsupervised_cursor_recalibration_offline', 'unsupervised_cursor_recalibration_online', 'willett_handwriting']


In [2]:
# Clone the public repo and import the reusable masked SSL helpers.

import os
import subprocess
import sys

REPO_URL = "https://github.com/ethan-read/utah-ssl.git"
REPO_DIR = Path("/content/utah-ssl")
EXPERIMENTS_DIR = REPO_DIR / "analysis" / "active" / "ssl_experiments"
MASKED_SSL_DIR = EXPERIMENTS_DIR / "masked_ssl"
SSL_DIR = REPO_DIR / "analysis" / "active" / "transfer_benchmark" / "ssl_autoresearch"

os.chdir("/content")

if REPO_DIR.exists():
    print("Using existing repo:", REPO_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

for candidate in (REPO_DIR, EXPERIMENTS_DIR, SSL_DIR):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

os.environ["SSL_AUTORESEARCH_CACHE_ROOT"] = str(CACHE_ROOT)
os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

if not MASKED_SSL_DIR.exists():
    raise FileNotFoundError(
        "The cloned repo does not contain analysis/active/ssl_experiments/masked_ssl. "
        "Make sure REPO_DIR points at a checkout that includes the masked reconstruction package."
    )

from masked_ssl import (
    CacheAccessConfig,
    DownstreamProbeConfig,
    SSLTrainingConfig,
    build_random_init_probe_state,
    build_segment_sampler,
    list_ssl_checkpoints,
    load_precomputed_session_feature_stats_into_cache_context,
    plot_ssl_training_history,
    prepare_cache_context,
    recover_downstream_probe_state,
    recover_ssl_run_state_from_checkpoint,
    resolve_ssl_checkpoint_path,
    run_checkpoint_probe_suite,
    run_downstream_probe,
    run_probe_head_sweep,
    resume_ssl_training,
    run_ssl_training,
    display_probe_summaries,
    display_ssl_reconstruction_report,
    SWEEP_VITAL_COLUMNS,
    run_sigma_mask_probe_sweep,
)
from masked_ssl.probe import build_downstream_probe_problem

print("cwd:", Path.cwd())
print("repo dir exists:", REPO_DIR.exists(), REPO_DIR)
print("experiments dir exists:", EXPERIMENTS_DIR.exists(), EXPERIMENTS_DIR)
print("masked_ssl dir exists:", MASKED_SSL_DIR.exists(), MASKED_SSL_DIR)
print("ssl dir exists:", SSL_DIR.exists(), SSL_DIR)
print("SSL_AUTORESEARCH_CACHE_ROOT:", os.environ["SSL_AUTORESEARCH_CACHE_ROOT"])
print("SSL_AUTORESEARCH_OUTPUT_ROOT:", os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"])


cwd: /content/utah-ssl
repo dir exists: True /content/utah-ssl
experiments dir exists: True /content/utah-ssl/analysis/active/ssl_experiments
masked_ssl dir exists: True /content/utah-ssl/analysis/active/ssl_experiments/masked_ssl
ssl dir exists: True /content/utah-ssl/analysis/active/transfer_benchmark/ssl_autoresearch
SSL_AUTORESEARCH_CACHE_ROOT: /content/drive/MyDrive/utah_ssl/data/cache_v1
SSL_AUTORESEARCH_OUTPUT_ROOT: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction


In [6]:
# Experiment config.

SEED = 7

SSL_STATE_MODE = "train"  # Set to "recover" to load a previous checkpoint.
GAUSSIAN_SMOOTHING_SIGMA_BINS = 2.0
SESSION_STATS_BIN_STRIDE = 2  # use every Nth bin when recomputing session stats
SESSION_STATS_VARIANT = "smoothed"  # one of {"smoothed", "unsmoothed", "none"}
STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED = Path("/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt")

def _sigma_tag_for_filename(value: float) -> str:
    value = float(value)
    text = f"{value:.6f}".rstrip("0").rstrip(".")
    if "." not in text:
        text = f"{text}.0"
    return text.replace("-", "m").replace(".", "p")

SMOOTHED_SIGMA_TAG = _sigma_tag_for_filename(GAUSSIAN_SMOOTHING_SIGMA_BINS)
STABLE_SSL_SESSION_STATS_PATH_SMOOTHED = Path(
    "/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/"
    f"session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma{SMOOTHED_SIGMA_TAG}_stable.pt"
)

if SESSION_STATS_VARIANT == "smoothed":
    STABLE_SSL_SESSION_STATS_PATH = STABLE_SSL_SESSION_STATS_PATH_SMOOTHED
elif SESSION_STATS_VARIANT == "unsmoothed":
    STABLE_SSL_SESSION_STATS_PATH = STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED
elif SESSION_STATS_VARIANT == "none":
    STABLE_SSL_SESSION_STATS_PATH = None
else:
    raise ValueError("SESSION_STATS_VARIANT must be one of {'smoothed', 'unsmoothed', 'none'}")

if STABLE_SSL_SESSION_STATS_PATH is not None and not STABLE_SSL_SESSION_STATS_PATH.exists():
    print(
        "warning: selected STABLE_SSL_SESSION_STATS_PATH does not exist; "
        "falling back to recomputing session stats during cache preparation:",
        STABLE_SSL_SESSION_STATS_PATH,
    )
    STABLE_SSL_SESSION_STATS_PATH = None

SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH = None
SSL_RECOVERY_RUN_DIR = None
ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH = None

FEATURE_MODE = "tx_only"
SEGMENT_BINS = 80
PATCH_SIZE = 4
PATCH_STRIDE = 2
HIDDEN_SIZE = 256
S5_STATE_SIZE = 128
NUM_LAYERS = 4
DROPOUT = 0
BATCH_SIZE = 32
NUM_STEPS = 1500
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0
VAL_EVERY = 50
VAL_BATCHES = 10
CHECKPOINT_EVERY_STEPS = 500
DATASET_WEIGHT_ALPHA = 0.25
EXAMPLES_PER_SHARD = 8
POST_PROJ_NORM = "rms"
RECONSTRUCTION_HEAD_TYPE = "mlp"
BACKBONE_DIRECTION = "bidirectional"

MASK_UNIT = "patch"  # Set to "bin" for raw-bin contiguous masking.
MASK_TOKEN_PLACEMENT = "before_projection"
MASK_RATIO = 0.4
PATCH_SPAN_LENGTH_MIN = 2
PATCH_SPAN_LENGTH_MAX = 4
BIN_SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN * PATCH_SIZE
BIN_SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX * PATCH_SIZE
SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MIN
SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MAX
NUM_SPANS_MODE = "multiple"
ALLOW_BIN_FRACTIONAL_OVERLAP = True
RECONSTRUCT_TARGET = "raw_patch"
LOSS_MODE = "masked_only"

CACHE_ACCESS_MODE = "drive_direct"  # or "copy_to_local"
LOCAL_CACHE_BASE = "/content/utah_ssl_cache"
FORCE_RECOPY_LOCAL_CACHE = False
EXCLUDED_DATASETS = {"brain2text25"}
LOG_EVERY = 10

NORMALIZE_IMPL_VERSION = "session_featurewise_v1"
CACHE_PREPARE_NORMALIZE_IMPL_VERSION = (
    "segment_prefix_v1"
    if (NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and STABLE_SSL_SESSION_STATS_PATH is not None)
    else NORMALIZE_IMPL_VERSION
)
NORMALIZE_CONTEXT_BINS = min(16, SEGMENT_BINS)
if NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and GAUSSIAN_SMOOTHING_SIGMA_BINS > 0 and SESSION_STATS_VARIANT == "unsmoothed":
    print(
        "warning: using smoothed inputs with unsmoothed session stats; "
        "set SESSION_STATS_VARIANT='smoothed' for matched preprocessing."
    )

CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode=CACHE_ACCESS_MODE,
    local_cache_base=LOCAL_CACHE_BASE,
    force_recopy_local_cache=FORCE_RECOPY_LOCAL_CACHE,
    excluded_datasets=tuple(sorted(EXCLUDED_DATASETS)),
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    normalize_context_bins=NORMALIZE_CONTEXT_BINS,
    normalize_impl_version=CACHE_PREPARE_NORMALIZE_IMPL_VERSION,
    gaussian_smoothing_sigma_bins=GAUSSIAN_SMOOTHING_SIGMA_BINS,
    session_stats_bin_stride=SESSION_STATS_BIN_STRIDE,
    examples_per_shard=EXAMPLES_PER_SHARD,
    feature_mode=FEATURE_MODE,
    boundary_key_mode="subject_if_available",
)

SSL_TRAINING_CONFIG = SSLTrainingConfig(
    seed=SEED,
    feature_mode=FEATURE_MODE,
    segment_bins=SEGMENT_BINS,
    patch_size=PATCH_SIZE,
    patch_stride=PATCH_STRIDE,
    hidden_size=HIDDEN_SIZE,
    s5_state_size=S5_STATE_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_every=VAL_EVERY,
    val_batches=VAL_BATCHES,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
    log_every=LOG_EVERY,
    post_proj_norm=POST_PROJ_NORM,
    reconstruction_head_type=RECONSTRUCTION_HEAD_TYPE,
    backbone_direction=BACKBONE_DIRECTION,
    boundary_key_mode="subject_if_available",
    mask_unit=MASK_UNIT,
    mask_token_placement=MASK_TOKEN_PLACEMENT,
    mask_ratio=MASK_RATIO,
    span_length_min=SPAN_LENGTH_MIN,
    span_length_max=SPAN_LENGTH_MAX,
    num_spans_mode=NUM_SPANS_MODE,
    reconstruct_target=RECONSTRUCT_TARGET,
    loss_mode=LOSS_MODE,
    allow_bin_fractional_overlap=ALLOW_BIN_FRACTIONAL_OVERLAP,
)

print("Notebook workflow switches:", {
    "SSL_STATE_MODE": SSL_STATE_MODE,
    "SESSION_STATS_VARIANT": SESSION_STATS_VARIANT,
    "STABLE_SSL_SESSION_STATS_PATH": STABLE_SSL_SESSION_STATS_PATH,
    "STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED": STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED,
    "STABLE_SSL_SESSION_STATS_PATH_SMOOTHED": STABLE_SSL_SESSION_STATS_PATH_SMOOTHED,
    "SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH": SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    "SSL_RECOVERY_RUN_DIR": SSL_RECOVERY_RUN_DIR,
    "ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH": ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH,
})
print("Masking config:", {
    "mask_unit": MASK_UNIT,
    "mask_token_placement": MASK_TOKEN_PLACEMENT,
    "mask_ratio": MASK_RATIO,
    "span_length_min": SPAN_LENGTH_MIN,
    "span_length_max": SPAN_LENGTH_MAX,
    "num_spans_mode": NUM_SPANS_MODE,
})
print("Patch config:", {
    "patch_size": PATCH_SIZE,
    "patch_stride": PATCH_STRIDE,
    "segment_bins": SEGMENT_BINS,
    "reconstruction_head_type": RECONSTRUCTION_HEAD_TYPE,
    "backbone_direction": BACKBONE_DIRECTION,
})
print("Feature config:", {
    "feature_mode": FEATURE_MODE,
    "normalize_impl_version": NORMALIZE_IMPL_VERSION,
    "cache_prepare_normalize_impl_version": CACHE_PREPARE_NORMALIZE_IMPL_VERSION,
    "gaussian_smoothing_sigma_bins": GAUSSIAN_SMOOTHING_SIGMA_BINS,
    "session_stats_bin_stride": SESSION_STATS_BIN_STRIDE,
})
print("CACHE_ACCESS_CONFIG:", CACHE_ACCESS_CONFIG)
print("SSL_TRAINING_CONFIG:", SSL_TRAINING_CONFIG)




Notebook workflow switches: {'SSL_STATE_MODE': 'train', 'SESSION_STATS_VARIANT': 'smoothed', 'STABLE_SSL_SESSION_STATS_PATH': PosixPath('/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma2p0_stable.pt'), 'STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED': PosixPath('/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt'), 'STABLE_SSL_SESSION_STATS_PATH_SMOOTHED': PosixPath('/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma2p0_stable.pt'), 'SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH': None, 'SSL_RECOVERY_RUN_DIR': None, 'ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH': None}
Masking config: {'mask_unit'

In [7]:
# Resolve cache access mode, summarize datasets, and build the reusable cache context.

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CACHE_CONTEXT = prepare_cache_context(
    cache_candidates=cache_candidates,
    config=CACHE_ACCESS_CONFIG,
)
FULL_DIM = CACHE_CONTEXT.full_dim

print("DEVICE:", DEVICE)
print("CACHE_CONTEXT cache_root:", CACHE_CONTEXT.cache_root)
print("FULL_DIM:", FULL_DIM)
print("session split summary:")
for dataset, summary in CACHE_CONTEXT.session_split_summary.items():
    print(
        f" - {dataset}: sessions={summary['total_sessions']} train_sessions={summary['train_sessions']} "
        f"val_sessions={summary['val_sessions']} train_examples={summary['train_examples']} "
        f"val_examples={summary['val_examples']}"
    )


using Drive-backed cache directly; skipping local copy
source: /content/drive/MyDrive/utah_ssl/data/cache_v1
source signature: 4331fe6a01f3
DEVICE: cuda
CACHE_CONTEXT cache_root: /content/drive/MyDrive/utah_ssl/data/cache_v1
FULL_DIM: 256
session split summary:
 - 000950: sessions=47 train_sessions=37 val_sessions=10 train_examples=569 val_examples=159
 - brain2text24: sessions=28 train_sessions=22 val_sessions=6 train_examples=12811 val_examples=3277
 - motor_data: sessions=21 train_sessions=16 val_sessions=5 train_examples=13611 val_examples=2809
 - plug_n_play: sessions=31 train_sessions=24 val_sessions=7 train_examples=1078 val_examples=237
 - unsupervised_cursor_recalibration_offline: sessions=103 train_sessions=82 val_sessions=21 train_examples=57000 val_examples=12157
 - unsupervised_cursor_recalibration_online: sessions=11 train_sessions=8 val_sessions=3 train_examples=72 val_examples=25
 - willett_handwriting: sessions=10 train_sessions=8 val_sessions=2 train_examples=4553 val

In [8]:
# Load precomputed SSL session-level featurewise z-scoring stats when available.

if STABLE_SSL_SESSION_STATS_PATH is not None:
    SSL_SESSION_STATS_STATE = load_precomputed_session_feature_stats_into_cache_context(
        cache_context=CACHE_CONTEXT,
        stats_path=STABLE_SSL_SESSION_STATS_PATH,
        normalize_impl_version=NORMALIZE_IMPL_VERSION,
    )
    print("Loaded cached SSL session stats from stable path.")
else:
    SSL_SESSION_STATS_STATE = {
        "stats_path": None,
        "metadata": {
            "source": "prepare_cache_context",
            "gaussian_smoothing_sigma_bins": float(GAUSSIAN_SMOOTHING_SIGMA_BINS),
            "session_stats_bin_stride": int(SESSION_STATS_BIN_STRIDE),
        },
        "session_count": len(CACHE_CONTEXT.session_feature_stats),
        "normalize_impl_version": CACHE_CONTEXT.normalize_impl_version,
    }
    print("Using session stats computed during cache preparation.")

print("session_count:", SSL_SESSION_STATS_STATE["session_count"])
print("normalize_impl_version:", SSL_SESSION_STATS_STATE["normalize_impl_version"])
print("metadata:", SSL_SESSION_STATS_STATE["metadata"])

print("expected_gaussian_smoothing_sigma_bins:", GAUSSIAN_SMOOTHING_SIGMA_BINS)
meta_sigma = SSL_SESSION_STATS_STATE["metadata"].get("gaussian_smoothing_sigma_bins")
print("stats_metadata_gaussian_smoothing_sigma_bins:", meta_sigma)
if meta_sigma is not None and abs(float(meta_sigma) - float(GAUSSIAN_SMOOTHING_SIGMA_BINS)) > 1e-6:
    print(
        "warning: loaded session stats metadata smoothing sigma does not match current config. "
        "This can skew normalization scale."
    )

print("expected_session_stats_bin_stride:", SESSION_STATS_BIN_STRIDE)
meta_stride = SSL_SESSION_STATS_STATE["metadata"].get("session_stats_bin_stride")
print("stats_metadata_session_stats_bin_stride:", meta_stride)
meta_stride_norm = None
if meta_stride is not None:
    try:
        meta_stride_norm = int(round(float(meta_stride)))
    except (TypeError, ValueError):
        print(
            "warning: loaded session stats metadata stride is not parseable as a number. "
            "This can skew normalization scale."
        )
if meta_stride_norm is not None and int(meta_stride_norm) != int(SESSION_STATS_BIN_STRIDE):
    print(
        "warning: loaded session stats metadata stride does not match current config. "
        "This can skew normalization scale."
    )





Loaded cached SSL session stats from stable path.
session_count: 251
normalize_impl_version: session_featurewise_v1
metadata: {'source': 's5_maskedreconstruction_in_memory', 'created_unix': 1776348862.3535125, 'normalize_impl_version': 'session_featurewise_v1', 'feature_mode': 'tx_only', 'gaussian_smoothing_sigma_bins': 2.0, 'session_stats_bin_stride': 1}
expected_gaussian_smoothing_sigma_bins: 2.0
stats_metadata_gaussian_smoothing_sigma_bins: 2.0
expected_session_stats_bin_stride: 2
stats_metadata_session_stats_bin_stride: 1


In [29]:
# # Quick batch inspection.

# INSPECT_SAMPLER = build_segment_sampler(
#     CACHE_CONTEXT,
#     "train",
#     batch_size=4,
#     seed=SEED,
#     segment_bins=SEGMENT_BINS,
#     dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
#     examples_per_shard=EXAMPLES_PER_SHARD,
# )
# inspect_batch = INSPECT_SAMPLER.sample_batch()
# print("inspect_batch[x].shape:", tuple(inspect_batch["x"].shape))
# print("inspect_batch[lengths]:", inspect_batch["lengths"].tolist())
# print("inspect_batch[datasets]:", inspect_batch["datasets"])
# print("inspect_batch[sessions]:", inspect_batch["sessions"])


In [32]:
# Acquire SSL state: either train now or recover from a saved checkpoint.

if NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and not CACHE_CONTEXT.session_feature_stats:
    raise RuntimeError("Run the stable session-stats cell first so session-level z-scoring is loaded into CACHE_CONTEXT.")

if SSL_STATE_MODE == "train":
    SSL_RUN_STATE = run_ssl_training(
        cache_context=CACHE_CONTEXT,
        config=SSL_TRAINING_CONFIG,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
    )
elif SSL_STATE_MODE == "recover":
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
else:
    raise ValueError(f"Unsupported SSL_STATE_MODE: {SSL_STATE_MODE}")

print("run_name:", SSL_RUN_STATE["run_name"])
print("run_dir:", SSL_RUN_STATE["run_dir"])
print("checkpoint_path:", SSL_RUN_STATE["checkpoint_path"])
print("best_score:", SSL_RUN_STATE["best_score"])
print("best_step:", SSL_RUN_STATE["best_step"])


step=001 train_loss=0.9549 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=78.0223 sample_s=0.07 model_s=0.41
step=010 train_loss=0.8735 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.0873 sample_s=0.03 model_s=0.40
step=020 train_loss=0.9426 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=0.9105 sample_s=0.03 model_s=0.41
step=030 train_loss=0.9575 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=0.8839 sample_s=0.04 model_s=0.42
step=040 train_loss=0.9106 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.3987 sample_s=0.04 model_s=0.41
step=050 train_loss=0.9281 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.1801 sample_s=0.03 model_s=0.41
step=050 val_loss=0.9909 val_mask_ratio=0.410 val_masked_token_ratio=0.410
step=060 train_loss=0.8994 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=0.8775 sample_s=0.03 model_s=0.41
step=070 train_loss=0.8941 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.2255 sample_s=0.03 model_s=0.41
step=080 train_loss=

In [16]:
# SSL RESUME cell: continue training from the current in-memory SSL_RUN_STATE.

RESUME_ADDITIONAL_STEPS = 1000

if "SSL_RUN_STATE" not in globals():
    if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is not None:
        resolved_checkpoint_path = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
        if not resolved_checkpoint_path.exists():
            raise FileNotFoundError(
                f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {resolved_checkpoint_path}"
            )
    else:
        resolved_checkpoint_path = resolve_ssl_checkpoint_path(
            output_root=OUTPUT_ROOT,
            explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
            run_dir=SSL_RECOVERY_RUN_DIR,
        )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

SSL_RUN_STATE = resume_ssl_training(
    run_state=SSL_RUN_STATE,
    additional_steps=RESUME_ADDITIONAL_STEPS,
    cache_context=CACHE_CONTEXT,
    device=DEVICE,
)


Continuing in-memory masked SSL training
 - run_dir: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260416T152031Z
 - start_step: 4000
 - target_step: 5000
 - mask_ratio: 0.15
step=4001 train_loss=0.4887 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1584 sample_s=0.04 model_s=0.41
step=4010 train_loss=0.4926 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1358 sample_s=0.03 model_s=0.40
step=4020 train_loss=0.4823 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1450 sample_s=0.04 model_s=0.40
step=4030 train_loss=0.5227 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.6602 sample_s=0.03 model_s=0.40
step=4040 train_loss=0.5019 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1851 sample_s=0.03 model_s=0.39
step=4050 train_loss=0.5199 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.2265 sample_s=0.04 model_s=0.41
step=4050 val_loss=0.7295 val_mask_ratio=0.154 val_masked_token_ratio=

In [37]:
# Resolve the active checkpoint path used by downstream probe recovery.

if "SSL_RUN_STATE" not in globals() and ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(SSL_RUN_STATE["checkpoint_path"])
else:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
    if not ACTIVE_SSL_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {ACTIVE_SSL_CHECKPOINT_PATH}"
        )

ACTIVE_SSL_CHECKPOINT_RUN_DIR = (
    ACTIVE_SSL_CHECKPOINT_PATH.parent.parent
    if ACTIVE_SSL_CHECKPOINT_PATH.parent.name == "checkpoints"
    else ACTIVE_SSL_CHECKPOINT_PATH.parent
)

print("ACTIVE_SSL_CHECKPOINT_PATH:", ACTIVE_SSL_CHECKPOINT_PATH)
print("ACTIVE_SSL_CHECKPOINT_RUN_DIR:", ACTIVE_SSL_CHECKPOINT_RUN_DIR)



ACTIVE_SSL_CHECKPOINT_PATH: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z/checkpoint_final.pt
ACTIVE_SSL_CHECKPOINT_RUN_DIR: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z


In [ ]:
# Plot train / val curves and print the compact SSL reconstruction scorecard.

if "SSL_RUN_STATE" not in globals():
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

SSL_RECON_SCORECARD = display_ssl_reconstruction_report(
    SSL_RUN_STATE,
    zero_baseline_df=globals().get("ZERO_BASELINE_DF"),
)


In [ ]:
# Optional: list saved SSL checkpoints for the active run.


CHECKPOINT_LIST_RUN_DIR = Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR)
AVAILABLE_SSL_CHECKPOINTS = list_ssl_checkpoints(CHECKPOINT_LIST_RUN_DIR)
display(pd.DataFrame(AVAILABLE_SSL_CHECKPOINTS))


## Downstream Probe Workflow

These cells keep the same held-out `Brain2Text25` workflow as the other `s5` notebooks.
The main apples-to-apples frozen comparison is:

- pretrained masked-reconstruction encoder, frozen, linear probe
- random-init encoder, frozen, same linear probe


In [ ]:
# Configure and preview the held-out Brain2Text25 probe split.

PROBE_SESSION_LIMIT = 8
PROBE_TARGET_SESSION_COUNT = 4
PROBE_BATCH_SIZE = 8
PROBE_BUDGET_SECONDS = 240
PROBE_MAX_STEPS = 80
PROBE_HEAD_LEARNING_RATE = 1e-3
ENCODER_LEARNING_RATE = 3e-4
PROBE_WEIGHT_DECAY = 1e-2
PROBE_INPUT_TAIL_BINS = 4  # 80 ms at 20 ms/bin

DOWNSTREAM_PROBE_CONFIG = DownstreamProbeConfig(
    enabled=True,
    seed=SEED,
    session_limit=PROBE_SESSION_LIMIT,
    target_session_count=PROBE_TARGET_SESSION_COUNT,
    probe_batch_size=PROBE_BATCH_SIZE,
    probe_budget_seconds=PROBE_BUDGET_SECONDS,
    max_probe_steps=PROBE_MAX_STEPS,
    probe_head_learning_rate=PROBE_HEAD_LEARNING_RATE,
    encoder_learning_rate=None,
    weight_decay=PROBE_WEIGHT_DECAY,
    probe_head_type="linear",
    probe_input_tail_bins=PROBE_INPUT_TAIL_BINS,
)

downstream_problem_preview = build_downstream_probe_problem(
    cache_root=Path(CACHE_CONTEXT.cache_root),
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    feature_mode=FEATURE_MODE,
)
print("eligible_sessions:", len(downstream_problem_preview["eligible_entries"]))
print("preview_source_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].train])
print("preview_target_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].val])
print("probe_input_tail_bins:", DOWNSTREAM_PROBE_CONFIG.probe_input_tail_bins)


eligible_sessions: 45
preview_source_sessions: ['t15.2024.07.21', 't15.2024.07.28', 't15.2025.01.10', 't15.2025.01.12']
preview_target_sessions: ['t15.2025.03.14', 't15.2025.03.16', 't15.2025.03.30', 't15.2025.04.13']


In [ ]:
# Shared downstream experiment helpers.

FROZEN_LINEAR_PROBE_OVERRIDES = {
    "probe_head_type": "linear",
    "probe_input_tail_bins": PROBE_INPUT_TAIL_BINS,
    "max_probe_steps": PROBE_MAX_STEPS,
}
FROZEN_CONV1D_PROBE_OVERRIDES = {
    "probe_head_type": "conv1d",
    "probe_input_tail_bins": PROBE_INPUT_TAIL_BINS,
    "max_probe_steps": PROBE_MAX_STEPS,
}
PROBE_HEAD_SPECS = [
    {"probe_head_type": "linear", "probe_input_tail_bins": PROBE_INPUT_TAIL_BINS},
    {"probe_head_type": "conv1d", "probe_input_tail_bins": PROBE_INPUT_TAIL_BINS},
]

def build_notebook_random_probe_state(*, seed_offset: int = 0):
    return build_random_init_probe_state(
        reference_config=DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_config"],
        input_dim=DOWNSTREAM_PROBE_DEFAULT_STATE["input_dim"],
        seed=SEED + int(seed_offset),
        base_run_dir=DOWNSTREAM_PROBE_BASE_RUN_DIR,
    )


In [38]:
# Recover the default SSL encoder and prepare reusable downstream probe state.

DOWNSTREAM_PROBE_DEFAULT_STATE = recover_downstream_probe_state(
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    output_root=OUTPUT_ROOT,
    input_dim=FULL_DIM,
    default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
    in_memory_model=SSL_RUN_STATE["model"] if "SSL_RUN_STATE" in globals() else None,
    current_checkpoint_path=Path(ACTIVE_SSL_CHECKPOINT_PATH),
    current_run_dir=Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR),
)

DOWNSTREAM_PROBE_BASE_RUN_DIR = Path(DOWNSTREAM_PROBE_DEFAULT_STATE["base_run_dir"])
print("DOWNSTREAM_PROBE_ENCODER_SOURCE:", DOWNSTREAM_PROBE_DEFAULT_STATE["source"])
print("DOWNSTREAM_PROBE_BASE_RUN_DIR:", DOWNSTREAM_PROBE_BASE_RUN_DIR)
print("DOWNSTREAM_PROBE_CHECKPOINT_PATH:", DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_path"])


DOWNSTREAM_PROBE_ENCODER_SOURCE: checkpoint
DOWNSTREAM_PROBE_BASE_RUN_DIR: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z
DOWNSTREAM_PROBE_CHECKPOINT_PATH: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z/checkpoint_final.pt


In [ ]:
# Frozen SSL checkpoint probes (linear + causal conv1d).

SSL_CHECKPOINT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint",
    artifact_prefix="ssl_checkpoint",
    train_encoder=False,
    head_specs=PROBE_HEAD_SPECS,
)

SSL_CHECKPOINT_LINEAR_SUMMARY = next(
    summary for summary in SSL_CHECKPOINT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)
SSL_CHECKPOINT_CONV1D_SUMMARY = next(
    summary for summary in SSL_CHECKPOINT_HEAD_SUMMARIES if summary.get("probe_head_type") == "conv1d"
)

display_probe_summaries(*SSL_CHECKPOINT_HEAD_SUMMARIES)


In [ ]:
# Random-init frozen probe baselines (linear + causal conv1d).

RANDOM_INIT_FROZEN_STATE = build_notebook_random_probe_state(seed_offset=1000)
RANDOM_INIT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=RANDOM_INIT_FROZEN_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init",
    artifact_prefix="random_init",
    train_encoder=False,
    head_specs=PROBE_HEAD_SPECS,
)

RANDOM_INIT_LINEAR_SUMMARY = next(
    summary for summary in RANDOM_INIT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)
RANDOM_INIT_CONV1D_SUMMARY = next(
    summary for summary in RANDOM_INIT_HEAD_SUMMARIES if summary.get("probe_head_type") == "conv1d"
)

display_probe_summaries(*SSL_CHECKPOINT_HEAD_SUMMARIES, *RANDOM_INIT_HEAD_SUMMARIES)


In [ ]:
# Brain2Text24 pooled val-session probes using the configured SSL split.

B2T24_PROBE_DATASET = "brain2text24"
if "B2T24_SSL_SPLIT_INFO" in globals():
    B2T24_CONFIG_TRAIN_SESSION_IDS = [str(session_id) for session_id in B2T24_SSL_SPLIT_INFO.get("train_session_ids", [])]
    B2T24_CONFIG_VAL_SESSION_IDS = [str(session_id) for session_id in B2T24_SSL_SPLIT_INFO.get("val_session_ids", [])]
else:
    split_summary = CACHE_CONTEXT.session_split_summary.get(B2T24_PROBE_DATASET, {})
    B2T24_CONFIG_TRAIN_SESSION_IDS = [str(session_id) for session_id in split_summary.get("train_session_ids", [])]
    B2T24_CONFIG_VAL_SESSION_IDS = [str(session_id) for session_id in split_summary.get("val_session_ids", [])]
    if not B2T24_CONFIG_VAL_SESSION_IDS:
        split_rows = CACHE_CONTEXT.split_rows_by_dataset
        train_rows = split_rows.get("train", {}).get(B2T24_PROBE_DATASET, [])
        val_rows = split_rows.get("val", {}).get(B2T24_PROBE_DATASET, [])
        B2T24_CONFIG_TRAIN_SESSION_IDS = sorted({str(row.session_id) for row in train_rows})
        B2T24_CONFIG_VAL_SESSION_IDS = sorted({str(row.session_id) for row in val_rows})

if not B2T24_CONFIG_VAL_SESSION_IDS:
    raise RuntimeError(
        "No Brain2Text24 val sessions were found in the active config split. "
        "Populate B2T24_SSL_SPLIT_INFO or run the split-config cell first."
    )

print("Brain2Text24 probe train sessions:", B2T24_CONFIG_TRAIN_SESSION_IDS)
print("Brain2Text24 probe val sessions:", B2T24_CONFIG_VAL_SESSION_IDS)

B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint_b2t24_config_val",
    artifact_prefix="ssl_checkpoint_b2t24_config_val",
    train_encoder=False,
    probe_dataset=B2T24_PROBE_DATASET,
    source_session_ids=B2T24_CONFIG_TRAIN_SESSION_IDS,
    target_session_ids=B2T24_CONFIG_VAL_SESSION_IDS,
    head_specs=PROBE_HEAD_SPECS,
)

B2T24_SSL_CHECKPOINT_LINEAR_SUMMARY = next(
    summary for summary in B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)
B2T24_SSL_CHECKPOINT_CONV1D_SUMMARY = next(
    summary for summary in B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES if summary.get("probe_head_type") == "conv1d"
)

B2T24_RANDOM_INIT_FROZEN_STATE = build_notebook_random_probe_state(seed_offset=2000)
B2T24_RANDOM_INIT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=B2T24_RANDOM_INIT_FROZEN_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init_b2t24_config_val",
    artifact_prefix="random_init_b2t24_config_val",
    train_encoder=False,
    probe_dataset=B2T24_PROBE_DATASET,
    source_session_ids=B2T24_CONFIG_TRAIN_SESSION_IDS,
    target_session_ids=B2T24_CONFIG_VAL_SESSION_IDS,
    head_specs=PROBE_HEAD_SPECS,
)

B2T24_RANDOM_INIT_LINEAR_SUMMARY = next(
    summary for summary in B2T24_RANDOM_INIT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)
B2T24_RANDOM_INIT_CONV1D_SUMMARY = next(
    summary for summary in B2T24_RANDOM_INIT_HEAD_SUMMARIES if summary.get("probe_head_type") == "conv1d"
)

display_probe_summaries(
    *B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES,
    *B2T24_RANDOM_INIT_HEAD_SUMMARIES,
)


In [ ]:
# Probe suite utilities for comparing multiple SSL checkpoints.

MAIN_SUMMARIES = [
    globals().get("SSL_CHECKPOINT_LINEAR_SUMMARY"),
    globals().get("SSL_CHECKPOINT_CONV1D_SUMMARY"),
    globals().get("RANDOM_INIT_LINEAR_SUMMARY"),
    globals().get("RANDOM_INIT_CONV1D_SUMMARY"),
]
display_probe_summaries(*MAIN_SUMMARIES)

CHECKPOINT_PROBE_PATHS = [Path(ACTIVE_SSL_CHECKPOINT_PATH)]

# Add extra checkpoints as needed, e.g.:
# CHECKPOINT_PROBE_PATHS = [
#     Path(ACTIVE_SSL_CHECKPOINT_PATH),
#     Path("/content/.../checkpoints/step_006000_...pt"),
#     Path("/content/.../checkpoints/step_008000_...pt"),
# ]

RUN_CHECKPOINT_PROBE_SUITE = False

if RUN_CHECKPOINT_PROBE_SUITE:
    CHECKPOINT_PROBE_SUMMARIES = run_checkpoint_probe_suite(
        checkpoint_paths=[Path(path) for path in CHECKPOINT_PROBE_PATHS],
        probe_config=DOWNSTREAM_PROBE_CONFIG,
        output_root=Path(OUTPUT_ROOT),
        input_dim=int(FULL_DIM),
        default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
        cache_root=Path(CACHE_CONTEXT.cache_root),
        device=DEVICE,
        head_specs=PROBE_HEAD_SPECS,
        variant_prefix="ssl_checkpoint",
        artifact_prefix="ssl_checkpoint",
        train_encoder=False,
    )
    CHECKPOINT_PROBE_DF = display_probe_summaries(*CHECKPOINT_PROBE_SUMMARIES)
    display(CHECKPOINT_PROBE_DF)


## Notes

This notebook intentionally omits the contrastive retrieval diagnostics from the older `s5` notebooks.
For this experiment, the primary scorecard is held-out frozen phoneme transfer, with masked-reconstruction loss used as a training diagnostic.


In [15]:
# Cell 1: Sweep + brain2text24 single-session probe setup

# Sweep controls
SWEEP_SIGMAS = [1.5, 2.0, 2.5]
SWEEP_MASK_RATIOS = [0.25]
SWEEP_SSL_STEPS = 1200

# Probe controls
PROBE_DATASET = "brain2text24"
PROBE_SESSION_ID = None  # set explicitly (e.g. "t12.2022.07.27") or leave None to auto-pick
PROBE_MAX_STEPS = 80
PROBE_BATCH_SIZE = 8
PROBE_HEAD_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-2
PROBE_TRAIN_ENCODER = False  # frozen encoder probe
PROBE_ALLOW_NONE_AS_TRAIN = False

# Training split (same toy split style you were using)
TRAIN_DATASET = globals().get("TARGET_DATASET", "brain2text24")
TRAIN_SESSION_ID = globals().get("TRAIN_SESSION_ID", "t12.2022.04.21")
VAL_SESSION_ID = globals().get("VAL_SESSION_ID", "t12.2022.07.27")

SKIP_SIGMA_IF_STATS_MISSING = True  # True => skip sigma with no precomputed stats
SESSION_STATS_DIR = Path("/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats")
REQUIRE_PRECOMPUTED_STATS_FOR_SWEEP = True


In [16]:
# Cell 2: Run the configured sigma x mask_ratio sweep and print compact vital statistics.

from IPython.display import display

SWEEP_RESULTS_DF, SWEEP_CONTEXTS_BY_SIGMA = run_sigma_mask_probe_sweep(
    sweep_sigmas=SWEEP_SIGMAS,
    sweep_mask_ratios=SWEEP_MASK_RATIOS,
    sweep_ssl_steps=SWEEP_SSL_STEPS,
    cache_candidates=cache_candidates,
    base_cache_config=CACHE_ACCESS_CONFIG,
    base_training_config=SSL_TRAINING_CONFIG,
    output_root=OUTPUT_ROOT,
    device=DEVICE,
    active_cache_context=globals().get("CACHE_CONTEXT"),
    active_sigma=float(globals().get("GAUSSIAN_SMOOTHING_SIGMA_BINS", float("nan"))),
    loaded_session_stats_state=globals().get("SSL_SESSION_STATS_STATE"),
    stable_stats_path=globals().get("STABLE_SSL_SESSION_STATS_PATH"),
    session_stats_dir=SESSION_STATS_DIR,
    normalize_impl_version=NORMALIZE_IMPL_VERSION,
    train_dataset=TRAIN_DATASET,
    train_session_id=TRAIN_SESSION_ID,
    val_session_id=VAL_SESSION_ID,
    probe_dataset=PROBE_DATASET,
    probe_session_id=PROBE_SESSION_ID,
    probe_max_steps=PROBE_MAX_STEPS,
    probe_batch_size=PROBE_BATCH_SIZE,
    probe_head_lr=PROBE_HEAD_LR,
    probe_weight_decay=PROBE_WEIGHT_DECAY,
    probe_train_encoder=PROBE_TRAIN_ENCODER,
    probe_allow_none_as_train=PROBE_ALLOW_NONE_AS_TRAIN,
    skip_sigma_if_stats_missing=SKIP_SIGMA_IF_STATS_MISSING,
    require_precomputed_stats_for_sweep=REQUIRE_PRECOMPUTED_STATS_FOR_SWEEP,
    boundary_key_mode="subject_if_available",
)

print("\nSweep complete.")
display(SWEEP_RESULTS_DF[SWEEP_VITAL_COLUMNS])


using Drive-backed cache directly; skipping local copy
source: /content/drive/MyDrive/utah_ssl/data/cache_v1
source signature: 4331fe6a01f3
[sigma=1.5] loaded stats: session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma1p5_stride2_stable.pt
[sigma=1.5] toy split: train=1320, val=682
[sigma=2.0] reusing existing CACHE_CONTEXT with preloaded session stats.
[sigma=2.0] toy split: train=1320, val=682
using Drive-backed cache directly; skipping local copy
source: /content/drive/MyDrive/utah_ssl/data/cache_v1
source signature: 4331fe6a01f3
[sigma=2.5] loaded stats: session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma2p5_stride2_stable.pt
[sigma=2.5] toy split: train=1320, val=682

=== sweep run: sigma=1.5, mask_ratio=0.25 ===
step=001 train_loss=0.9474 mask_ratio=0.256 masked_token_ratio=0.256 grad_norm=77.9586 sample_s=0.07 model_s=2.28
step=010 train_loss=0.9024 mask_ratio=0.256 masked_token_ratio=0.256 grad_nor

,sigma,mask_ratio,ssl_steps,val_model_masked_mse,val_masked_target_std,val_masked_prediction_std,val_prediction_target_corr,downstream_ctc_bpphone,downstream_per,reference_output_len,actual_output_len,actual_over_reference_len,most_common_prediction,most_common_prediction_rate
0,2.0,0.25,1200,0.888235,0.983381,0.441014,0.315341,5.167542,0.728452,1079,650,0.602410,SIL,0.506154
1,1.5,0.25,1200,0.895177,0.977685,0.403659,0.283666,5.304836,0.751622,1079,584,0.541242,SIL,0.534247
2,2.5,0.25,1200,0.882870,0.993217,0.472217,0.348438,5.525782,0.760890,1079,883,0.818350,SIL,0.387316


In [ ]:
# EXPERIMENT: Brain2Text24 chronological 12-session split (9 train / 3 val).

import copy
from datetime import datetime

import pandas as pd

TARGET_DATASET = "brain2text24"
B2T24_SPLIT_TOTAL_SESSIONS = 12
B2T24_VAL_SESSION_COUNT = 3
EXPECTED_LATEST_12_SESSIONS = [
    "t12.2022.07.05",
    "t12.2022.07.07",
    "t12.2022.07.14",
    "t12.2022.07.21",
    "t12.2022.07.27",
    "t12.2022.07.29",
    "t12.2022.08.02",
    "t12.2022.08.11",
    "t12.2022.08.13",
    "t12.2022.08.18",
    "t12.2022.08.23",
    "t12.2022.08.25",
]

if "CACHE_CONTEXT" not in globals():
    raise RuntimeError("CACHE_CONTEXT missing. Run the cache preparation cell first.")
if TARGET_DATASET not in CACHE_CONTEXT.rows_by_dataset:
    raise RuntimeError(f"{TARGET_DATASET!r} was not found in CACHE_CONTEXT.rows_by_dataset.")

if "_ORIG_CACHE_SPLITS" not in globals():
    _ORIG_CACHE_SPLITS = copy.deepcopy(CACHE_CONTEXT.split_rows_by_dataset)
    _ORIG_ROWS_BY_DATASET = copy.deepcopy(CACHE_CONTEXT.rows_by_dataset)
    _ORIG_PRETRAIN_DATASETS = list(CACHE_CONTEXT.pretrain_datasets)
    _ORIG_HAS_VAL_DATASETS = bool(CACHE_CONTEXT.has_val_datasets)
    _ORIG_SESSION_SPLIT_SUMMARY = copy.deepcopy(CACHE_CONTEXT.session_split_summary)

def _session_sort_key(session_id: str) -> tuple[datetime, str]:
    try:
        date_part = session_id.split(".", 1)[1]
        return datetime.strptime(date_part, "%Y.%m.%d"), session_id
    except Exception:
        return datetime.min, session_id

all_rows = list(_ORIG_ROWS_BY_DATASET[TARGET_DATASET])
all_session_ids = sorted({row.session_id for row in all_rows}, key=_session_sort_key)
if len(all_session_ids) < B2T24_SPLIT_TOTAL_SESSIONS:
    raise RuntimeError(
        f"Need at least {B2T24_SPLIT_TOTAL_SESSIONS} sessions, found {len(all_session_ids)}."
    )

latest_sessions = all_session_ids[-B2T24_SPLIT_TOTAL_SESSIONS:]
if latest_sessions != EXPECTED_LATEST_12_SESSIONS:
    raise RuntimeError(
        "Latest 12 sessions did not match the expected deterministic list. "
        f"expected={EXPECTED_LATEST_12_SESSIONS} actual={latest_sessions}"
    )

train_session_ids = latest_sessions[:-B2T24_VAL_SESSION_COUNT]
val_session_ids = latest_sessions[-B2T24_VAL_SESSION_COUNT:]
selected_sessions = set(latest_sessions)

train_rows = [row for row in all_rows if row.session_id in set(train_session_ids)]
val_rows = [row for row in all_rows if row.session_id in set(val_session_ids)]
if not train_rows or not val_rows:
    raise RuntimeError("Train/val rows are empty after applying the 12-session split.")

CACHE_CONTEXT.pretrain_datasets = [TARGET_DATASET]
CACHE_CONTEXT.split_rows_by_dataset["train"][TARGET_DATASET] = train_rows
CACHE_CONTEXT.split_rows_by_dataset["val"][TARGET_DATASET] = val_rows
CACHE_CONTEXT.rows_by_dataset[TARGET_DATASET] = [
    row for row in all_rows if row.session_id in selected_sessions
]

train_counts = {
    session_id: sum(1 for row in train_rows if row.session_id == session_id)
    for session_id in train_session_ids
}
val_counts = {
    session_id: sum(1 for row in val_rows if row.session_id == session_id)
    for session_id in val_session_ids
}

CACHE_CONTEXT.session_split_summary[TARGET_DATASET] = {
    "total_sessions": len(latest_sessions),
    "train_sessions": len(train_session_ids),
    "val_sessions": len(val_session_ids),
    "val_eligible": True,
    "train_examples": len(train_rows),
    "val_examples": len(val_rows),
    "train_session_ids": list(train_session_ids),
    "val_session_ids": list(val_session_ids),
}
CACHE_CONTEXT.has_val_datasets = True
CACHE_CONTEXT.sampling_plan_cache.clear()

B2T24_SSL_SPLIT_INFO = {
    "dataset": TARGET_DATASET,
    "selected_sessions": list(latest_sessions),
    "train_session_ids": list(train_session_ids),
    "val_session_ids": list(val_session_ids),
    "train_rows": len(train_rows),
    "val_rows": len(val_rows),
    "train_counts": train_counts,
    "val_counts": val_counts,
}

split_rows = []
for sid in train_session_ids:
    split_rows.append({"split": "train", "session_id": sid, "rows": train_counts[sid]})
for sid in val_session_ids:
    split_rows.append({"split": "val", "session_id": sid, "rows": val_counts[sid]})

split_df = pd.DataFrame(split_rows)
display(split_df)
print(
    "Configured Brain2Text24 split:",
    f"train_sessions={len(train_session_ids)} val_sessions={len(val_session_ids)}",
)
print("Total rows:", f"train={len(train_rows)}", f"val={len(val_rows)}")


In [ ]:
# EXPERIMENT: medium-scale Brain2Text24 SSL config (sigma=2.5, mask=0.25, 10k steps).

from dataclasses import replace
from pathlib import Path

TARGET_DATASET = "brain2text24"
GAUSSIAN_SMOOTHING_SIGMA_BINS = 2.5
MASK_RATIO = 0.25
NUM_STEPS = 10_000
VAL_EVERY = 100
CHECKPOINT_EVERY_STEPS = 1_000
LOG_EVERY = 25
SESSION_STATS_VARIANT = "smoothed"

if "CACHE_ACCESS_CONFIG" not in globals() or "SSL_TRAINING_CONFIG" not in globals():
    raise RuntimeError("CACHE_ACCESS_CONFIG/SSL_TRAINING_CONFIG missing. Run base config cell first.")
if "CACHE_CONTEXT" not in globals():
    raise RuntimeError("CACHE_CONTEXT missing. Run cache preparation cell first.")

def _sigma_tag_for_filename(value: float) -> str:
    text = f"{float(value):.6f}".rstrip("0").rstrip(".")
    if "." not in text:
        text = f"{text}.0"
    return text.replace("-", "m").replace(".", "p")

stride = int(globals().get("SESSION_STATS_BIN_STRIDE", CACHE_ACCESS_CONFIG.session_stats_bin_stride))
sigma_tag = _sigma_tag_for_filename(GAUSSIAN_SMOOTHING_SIGMA_BINS)
SESSION_STATS_DIR = Path(
    "/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats"
)
stats_candidates = [
    SESSION_STATS_DIR
    / f"session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma{sigma_tag}_stride{stride}_stable.pt",
    SESSION_STATS_DIR
    / f"session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma{sigma_tag}_stable.pt",
]
STABLE_SSL_SESSION_STATS_PATH = next((p for p in stats_candidates if p.exists()), None)

CACHE_PREPARE_NORMALIZE_IMPL_VERSION = (
    "segment_prefix_v1" if STABLE_SSL_SESSION_STATS_PATH is not None else CACHE_ACCESS_CONFIG.normalize_impl_version
)

CACHE_ACCESS_CONFIG = replace(
    CACHE_ACCESS_CONFIG,
    gaussian_smoothing_sigma_bins=float(GAUSSIAN_SMOOTHING_SIGMA_BINS),
    normalize_impl_version=str(CACHE_PREPARE_NORMALIZE_IMPL_VERSION),
)
SSL_TRAINING_CONFIG = replace(
    SSL_TRAINING_CONFIG,
    mask_ratio=float(MASK_RATIO),
    num_steps=int(NUM_STEPS),
    val_every=int(VAL_EVERY),
    checkpoint_every_steps=int(CHECKPOINT_EVERY_STEPS),
    log_every=int(LOG_EVERY),
)

CACHE_CONTEXT.config.gaussian_smoothing_sigma_bins = float(GAUSSIAN_SMOOTHING_SIGMA_BINS)
CACHE_CONTEXT.config.normalize_impl_version = str(CACHE_PREPARE_NORMALIZE_IMPL_VERSION)

if STABLE_SSL_SESSION_STATS_PATH is not None:
    _ = load_precomputed_session_feature_stats_into_cache_context(
        cache_context=CACHE_CONTEXT,
        stats_path=Path(STABLE_SSL_SESSION_STATS_PATH),
        normalize_impl_version="session_featurewise_v1",
    )
    print("Loaded precomputed session stats:", STABLE_SSL_SESSION_STATS_PATH.name)
else:
    print("warning: no precomputed sigma=2.5 stats found; using current in-memory session stats.")

CACHE_CONTEXT.sampling_plan_cache.clear()

print("Config sanity checks:")
print(" - cache_context.pretrain_datasets:", CACHE_CONTEXT.pretrain_datasets)
print(" - sigma:", CACHE_CONTEXT.gaussian_smoothing_sigma_bins)
print(" - mask_ratio:", SSL_TRAINING_CONFIG.mask_ratio)
print(" - num_steps:", SSL_TRAINING_CONFIG.num_steps)
print(" - val_every:", SSL_TRAINING_CONFIG.val_every)
print(" - checkpoint_every_steps:", SSL_TRAINING_CONFIG.checkpoint_every_steps)
print(" - log_every:", SSL_TRAINING_CONFIG.log_every)


In [ ]:
# REPORT: post-run summary + leakage checks for the Brain2Text24 medium-scale run.

import matplotlib.pyplot as plt
import pandas as pd

run_state = globals().get("SSL_RUN_STATE")
if run_state is None:
    raise RuntimeError("SSL_RUN_STATE not found. Run the SSL training cell first.")
if "B2T24_SSL_SPLIT_INFO" not in globals():
    raise RuntimeError("B2T24_SSL_SPLIT_INFO missing. Run the chronological split cell first.")

train_history = list(run_state.get("train_history", []))
val_history = list(run_state.get("val_history", []))
if not train_history:
    raise RuntimeError("SSL_RUN_STATE has no train_history.")

train_final = train_history[-1]
train_best = min(train_history, key=lambda record: float(record.get("loss", float("inf"))))
val_final = val_history[-1] if val_history else None
val_best = (
    min(val_history, key=lambda record: float(record.get("loss", float("inf"))))
    if val_history
    else None
)

print("Run dir:", run_state.get("run_dir"))
print("Checkpoint final:", run_state.get("checkpoint_path"))
print("Best checkpoint:", run_state.get("best_checkpoint_path"))
print("")
print("Masked MSE summary:")
print(f" - train final: {float(train_final['loss']):.6f} (step {int(train_final['step'])})")
print(f" - train best : {float(train_best['loss']):.6f} (step {int(train_best['step'])})")
if val_final is not None:
    print(f" - val final  : {float(val_final['loss']):.6f} (step {int(val_final['step'])})")
if val_best is not None:
    print(f" - val best   : {float(val_best['loss']):.6f} (step {int(val_best['step'])})")

plt.figure(figsize=(8, 5))
train_steps = [int(record["step"]) for record in train_history]
train_losses = [float(record["loss"]) for record in train_history]
plt.plot(train_steps, train_losses, label="train loss", alpha=0.85)
if val_history:
    val_steps = [int(record["step"]) for record in val_history]
    val_losses = [float(record["loss"]) for record in val_history]
    plt.plot(val_steps, val_losses, marker="o", label="val loss")
plt.xlabel("Step")
plt.ylabel("Masked MSE")
plt.title("Brain2Text24 SSL Loss Curve")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

target_dataset = str(B2T24_SSL_SPLIT_INFO.get("dataset", globals().get("TARGET_DATASET", "brain2text24")))
train_session_ids = list(B2T24_SSL_SPLIT_INFO.get("train_session_ids", []))
val_session_ids = list(B2T24_SSL_SPLIT_INFO.get("val_session_ids", []))

overlap = sorted(set(train_session_ids) & set(val_session_ids))
print("Leakage check:")
print(" - train session count:", len(train_session_ids))
print(" - val session count:", len(val_session_ids))
print(" - overlap:", overlap)
if overlap:
    raise RuntimeError(f"Leakage detected: overlapping sessions {overlap}")

train_rows = list(CACHE_CONTEXT.split_rows_by_dataset["train"].get(target_dataset, []))
val_rows = list(CACHE_CONTEXT.split_rows_by_dataset["val"].get(target_dataset, []))

counts_rows = []
for sid in train_session_ids:
    counts_rows.append(
        {
            "split": "train",
            "session_id": sid,
            "rows": sum(1 for row in train_rows if row.session_id == sid),
        }
    )
for sid in val_session_ids:
    counts_rows.append(
        {
            "split": "val",
            "session_id": sid,
            "rows": sum(1 for row in val_rows if row.session_id == sid),
        }
    )
counts_df = pd.DataFrame(counts_rows)
display(counts_df)

if (counts_df["rows"] <= 0).any():
    raise RuntimeError("At least one selected train/val session has zero rows in CACHE_CONTEXT splits.")
